# Module 19 — Sampling Strategies: Temperature, Top-k, Top-p, Min-p

**Part V · Generation & Alignment · 30–35 min**

---

In Module 18 we put a microscope on the model's last step: the logits get a softmax slapped on them and out comes a probability distribution over ~50,000 tokens. We stared at it. We marvelled. We moved on.

But staring isn't generating. *Somebody* has to actually pick a token. That somebody is the **sampler**, and it is — without exaggeration — the most under-thought-about part of the entire LLM stack.

People will spend three weeks fine-tuning a 7B model and then ship it with `temperature=0.7` because that's what was in the example code. Three weeks of gradient descent, undone by a magic number nobody questioned.

This notebook is a small intervention. We're going to:

1. Take *one* concrete next-token distribution from GPT-2 and reshape it five different ways
2. Walk through the math of each strategy — temperature, top-k, top-p, min-p — with pictures
3. Generate 200-token continuations under each and *measure* the differences (entropy, n-gram repetition, vibes)
4. Break things on purpose: T=0 on an open-ended prompt (watch it loop), T=2.0 (watch it dissolve)
5. End with a strong opinion about what you should actually use in 2026

The thesis: **the sweet spot is narrow and task-dependent, and the defaults you inherited are usually wrong**.

## 0 · Setup

Plain PyTorch, HuggingFace `transformers` for GPT-2, matplotlib for plots. Runs comfortably on CPU — sampling is cheap; we're not training anything.

In [ ]:
import math
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import GPT2LMHeadModel, GPT2TokenizerFast

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)

In [ ]:
# Same palette as Parts I-IV.
PALETTE = {
    "ink":     "#1a1a2e",
    "paper":   "#f7f3e9",
    "rose":    "#e63946",
    "amber":   "#f4a261",
    "teal":    "#2a9d8f",
    "indigo":  "#3d5a80",
    "plum":    "#7b2cbf",
    "lime":    "#a8dadc",
}

plt.rcParams.update({
    "figure.facecolor": PALETTE["paper"],
    "axes.facecolor":   PALETTE["paper"],
    "axes.edgecolor":   PALETTE["ink"],
    "axes.labelcolor":  PALETTE["ink"],
    "xtick.color":      PALETTE["ink"],
    "ytick.color":      PALETTE["ink"],
    "text.color":       PALETTE["ink"],
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("palette loaded")

In [ ]:
# Load GPT-2 small. Tiny by 2026 standards, but its sampler behavior is
# qualitatively identical to a 70B model — the math doesn't care about scale.
tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()
VOCAB = tok.vocab_size
print(f"GPT-2 loaded · vocab size: {VOCAB:,}")

## 1 · One distribution to rule them all

Module 18 ended with a next-token distribution. Let's reproduce one and use the *same one* for every plot in sections 2-5. That way every chart is comparable: same prompt, same logits, only the sampling rule changes.

We'll use a deliberately mid-context prompt — not so constraining that the answer is forced (where any sampler works fine), not so open that the distribution is flat (where any sampler behaves the same). Right in the middle is where sampling decisions actually matter.

In [ ]:
PROMPT = "The detective walked into the dim warehouse and saw"
input_ids = tok(PROMPT, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    logits = model(input_ids).logits[0, -1]   # (vocab,)
logits = logits.float().cpu()
probs = F.softmax(logits, dim=-1).numpy()

# Entropy of the raw distribution, in bits.
H = -(probs * np.log2(probs + 1e-12)).sum()
print(f"entropy (bits): {H:.2f}  (uniform over 50k = {math.log2(VOCAB):.2f})")
print(f"max probability: {probs.max():.3f}")
print(f"top-1 token: {tok.decode([int(probs.argmax())])!r}")

# Show the top 10 — this is the "raw material" every sampler will reshape.
top_idx = np.argsort(-probs)[:10]
for i, idx in enumerate(top_idx):
    print(f"  {i+1:2d}. {tok.decode([int(idx)])!r:>15}  p={probs[idx]:.4f}")

Entropy of ~5-7 bits over a 50k vocabulary is a good "open but not chaotic" distribution. The top token is plausible but only takes maybe 15-25% of the mass. There are dozens of reasonable continuations.

This is the *interesting* regime. If the top token had p=0.99, sampling wouldn't matter — every sane strategy picks it. If the distribution were uniform, every sampler would just produce noise. The middle is where samplers earn their keep.

In [ ]:
# Visualize the top-50 tokens. We plot the raw distribution once here
# so you can hold it in your head — every reshape later is a deformation of THIS.
TOPN = 50
top_idx = np.argsort(-probs)[:TOPN]
top_p = probs[top_idx]
top_labels = [tok.decode([int(i)]).strip() or "·" for i in top_idx]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(TOPN), top_p, color=PALETTE["indigo"], edgecolor=PALETTE["ink"], lw=0.5)
ax.set_xticks(range(TOPN))
ax.set_xticklabels(top_labels, rotation=70, ha="right", fontsize=8)
ax.set_ylabel("P(token)")
ax.set_title(f"Raw next-token distribution after: '{PROMPT}'  ·  H = {H:.2f} bits")
plt.tight_layout()
plt.show()

## 2 · Temperature — the volume knob

The simplest knob, and the one everybody adjusts first.

$$P_T(x) = \mathrm{softmax}(z / T)$$

where $z$ are the raw logits. That's it. You divide before softmax.

What does this *do*?

- **$T = 1$**: no change. Standard softmax. The model's "honest" distribution.
- **$T < 1$**: divisions make logits *bigger* in absolute terms → softmax sharpens → mass piles onto the top tokens. As $T \to 0$, $P_T$ becomes a delta at the argmax (greedy decoding).
- **$T > 1$**: logits shrink → softmax flattens → mass spreads across more tokens. As $T \to \infty$, $P_T$ approaches uniform over the entire vocabulary.

Two things people forget:

1. Temperature **doesn't change rankings**. The top token at T=0.1 is the same as at T=2.0. It just changes how *concentrated* the distribution is around it.
2. Temperature **affects every token in the vocabulary**, including the long tail. At T=2.0 you are *actively giving probability* to tokens the model thinks are nonsense. This is the source of "T=2.0 is gibberish" — you're sampling from a 50,000-element near-uniform.

In [ ]:
def apply_temperature(logits, T):
    if T == 0:
        # Greedy: a delta at the argmax.
        out = torch.full_like(logits, -1e9)
        out[logits.argmax()] = 0.0
        return F.softmax(out, dim=-1)
    return F.softmax(logits / T, dim=-1)

TEMPS = [0.1, 0.5, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(TEMPS), figsize=(15, 3.2), sharey=True)
colors = [PALETTE["rose"], PALETTE["amber"], PALETTE["indigo"], PALETTE["teal"], PALETTE["plum"]]

for ax, T, c in zip(axes, TEMPS, colors):
    p_T = apply_temperature(logits, T).numpy()
    top_T = p_T[top_idx]   # use the same top-50 ordering as section 1
    ax.bar(range(TOPN), top_T, color=c, edgecolor=PALETTE["ink"], lw=0.3)
    H_T = -(p_T * np.log2(p_T + 1e-12)).sum()
    ax.set_title(f"T = {T}\nH = {H_T:.2f} bits", fontsize=10)
    ax.set_xticks([])
    ax.set_xlabel("top-50 tokens")
axes[0].set_ylabel("P(token)")
plt.suptitle("Temperature reshapes the same distribution", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Another way to see what temperature does: plot the cumulative distribution
# of the sorted probabilities. A sharp distribution rises fast; a flat one
# rises slowly. Temperature literally rotates this curve.
fig, ax = plt.subplots(figsize=(8, 4.2))
for T, c in zip(TEMPS, colors):
    p_T = apply_temperature(logits, T).numpy()
    sorted_p = np.sort(p_T)[::-1]
    cum = np.cumsum(sorted_p)
    ax.plot(np.arange(1, 201), cum[:200], lw=2, color=c, label=f"T={T}")
ax.axhline(0.9, ls="--", color=PALETTE["ink"], alpha=0.4, lw=1)
ax.text(180, 0.91, "p=0.9", fontsize=9, color=PALETTE["ink"])
ax.set_xlabel("rank (sorted by probability)")
ax.set_ylabel("cumulative probability")
ax.set_title("Cumulative distribution at different temperatures")
ax.legend(loc="lower right")
ax.set_xscale("log")
plt.tight_layout()
plt.show()

Read those entropy numbers. At T=0.1 the distribution is essentially a single spike — entropy collapses near zero. At T=2.0 the entropy starts climbing toward the uniform ceiling of ~15.6 bits. T=1 is the model's honest opinion; everything else is you bullying the distribution.

Notice also the *long tail*. Look closely at the tiny bars beyond the top 10 in the T=1.5 and T=2.0 panels — those are tokens the model thought were extremely unlikely, now boosted into "actually plausible" range. That's where gibberish creeps in.

## 3 · Top-k — chop the tail

Top-k is the bluntest possible fix for "the tail is garbage": just delete it.

$$P_{\text{top-}k}(x) = \begin{cases} P(x) / Z & x \in \text{top-}k \text{ tokens} \\ 0 & \text{otherwise} \end{cases}$$

where $Z$ is the renormalizing constant. You keep the $k$ highest-probability tokens, zero out everything else, then divide so it sums to 1.

This works *kind of* fine. The problem is $k$ is **fixed**. If the distribution is sharp (the model is confident — say, the next token after "the United States of") your top-50 contains the right answer plus 49 distractors that would never have been sampled anyway. If the distribution is flat (the model is in a creative writing flow) your top-50 brutally cuts off legitimate continuations.

Top-k cannot adapt to how confident the model is. That's why top-p was invented in 2019.

In [ ]:
def apply_top_k(probs_np, k):
    out = np.zeros_like(probs_np)
    idx = np.argpartition(-probs_np, k)[:k]
    out[idx] = probs_np[idx]
    out /= out.sum()
    return out

KS = [5, 20, 100]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, k in zip(axes, KS):
    p_k = apply_top_k(probs, k)
    top_show = p_k[top_idx]
    ax.bar(range(TOPN), top_show, color=PALETTE["teal"], edgecolor=PALETTE["ink"], lw=0.3)
    kept = (p_k > 0).sum()
    ax.set_title(f"top-k, k = {k}\nkept {kept} tokens", fontsize=10)
    ax.set_xlabel("top-50 tokens (raw ordering)")
    ax.set_xticks([])
axes[0].set_ylabel("P(token)")
plt.suptitle("Top-k: blunt instrument, fixed cutoff", y=1.05)
plt.tight_layout()
plt.show()

See what happened with k=5? Five bars and then a cliff. The renormalization redistributes mass among the survivors, so they get *bigger* than they were originally. That's the "sharpening" side-effect of top-k that nobody mentions.

## 4 · Top-p (nucleus) — the *adaptive* tail-chop

Top-p (Holtzman et al., 2019) fixed the fundamental problem with top-k. Instead of "keep $k$ tokens", it says **"keep the smallest set of tokens whose cumulative probability is at least $p$"**.

$$\text{Nucleus}_p = \text{smallest } V' \subseteq V \text{ such that } \sum_{x \in V'} P(x) \geq p$$

Then renormalize over the nucleus.

The crucial property: the *size* of the nucleus depends on the distribution.

- **Sharp distribution** (the model is confident): a few tokens already cover 90% of mass. The nucleus is tiny — maybe 2 or 3 tokens.
- **Flat distribution** (the model is uncertain): you need many tokens to accumulate 90% mass. The nucleus might be 200+ tokens.

So top-p is "loose when the model is uncertain, strict when the model is sure". That's exactly what you want.

In [ ]:
def apply_top_p(probs_np, p):
    sorted_idx = np.argsort(-probs_np)
    sorted_p = probs_np[sorted_idx]
    cum = np.cumsum(sorted_p)
    # smallest set whose cumulative prob >= p
    cutoff = int(np.searchsorted(cum, p)) + 1
    keep_idx = sorted_idx[:cutoff]
    out = np.zeros_like(probs_np)
    out[keep_idx] = probs_np[keep_idx]
    out /= out.sum()
    return out, cutoff

# Demonstrate the dynamic nature: same p=0.9 against TWO different distributions.
# Distribution A: our prompt (mid-entropy, the "warehouse" prompt).
# Distribution B: a sharp distribution — finishing a fixed phrase.
PROMPT_B = "The capital of France is"
ids_B = tok(PROMPT_B, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logits_B = model(ids_B).logits[0, -1].float().cpu()
probs_B = F.softmax(logits_B, dim=-1).numpy()

p = 0.9
_, k_A = apply_top_p(probs, p)
_, k_B = apply_top_p(probs_B, p)
print(f"Same p = {p}")
print(f"  prompt A ('warehouse...')   → nucleus = {k_A:>4} tokens")
print(f"  prompt B ('capital of...') → nucleus = {k_B:>4} tokens")
print()
print("Same threshold, vastly different cutoff — that's the whole point of top-p.")

In [ ]:
# Plot top-p reshaping for several values of p, on prompt A.
PS = [0.5, 0.9, 0.95, 0.99]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, p in zip(axes, PS):
    p_p, k = apply_top_p(probs, p)
    top_show = p_p[top_idx]
    ax.bar(range(TOPN), top_show, color=PALETTE["plum"], edgecolor=PALETTE["ink"], lw=0.3)
    ax.set_title(f"top-p, p = {p}\nnucleus = {k} tokens", fontsize=10)
    ax.set_xticks([])
    ax.set_xlabel("top-50 tokens")
axes[0].set_ylabel("P(token)")
plt.suptitle("Top-p adapts: nucleus grows as p increases", y=1.05)
plt.tight_layout()
plt.show()

The pathology of top-p (which top-p evangelists usually skip): on a *very* flat distribution, the nucleus can swell to thousands of tokens, and you're effectively sampling from raw GPT-2. On a *very* sharp distribution, the nucleus collapses to one token, and you've accidentally made it greedy. The dynamic range is wide, which is great until it isn't.

## 5 · Min-p — the 2025 default

Min-p (Nguyen et al., 2024) is the newest member of the family and as of 2026 it is the **default in vLLM, Aphrodite, and basically every serious inference engine** that gives you a choice. The idea:

$$P_{\text{min-}p}(x) = \begin{cases} P(x)/Z & P(x) \geq p_{\min} \cdot P(x_{\max}) \\ 0 & \text{otherwise} \end{cases}$$

where $x_{\max}$ is the most-likely token. In English: **"keep any token whose probability is at least some fraction of the top token's probability"**.

Why is this so much better than top-p?

- **Sharp distribution**: top token has p=0.9, second has p=0.05. With $p_{\min}=0.1$, threshold = 0.09. We keep only the top token. Correct: the model is confident.
- **Flat distribution**: top token has p=0.02, second has p=0.018, third has p=0.017... With $p_{\min}=0.1$, threshold = 0.002. We keep the top *several hundred* tokens. Correct: the model is uncertain, give it room.
- **The bimodal failure case** that *kills* top-p: top token p=0.5, next 99 tokens each p=0.005. Top-p with p=0.95 keeps all 100 (because cumulative needs to hit 0.95). Min-p with $p_{\min}=0.1$ keeps only the top token (threshold 0.05, only the leader passes). Min-p makes the *correct* decision; top-p degenerates.

In practice $p_{\min} \in [0.05, 0.1]$ works for almost everything. It is the closest thing to a "set it and forget it" sampler.

In [ ]:
def apply_min_p(probs_np, p_min):
    threshold = p_min * probs_np.max()
    out = np.where(probs_np >= threshold, probs_np, 0.0)
    out /= out.sum()
    return out, int((out > 0).sum())

PMINS = [0.01, 0.05, 0.1, 0.2]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, pm in zip(axes, PMINS):
    p_m, k = apply_min_p(probs, pm)
    top_show = p_m[top_idx]
    ax.bar(range(TOPN), top_show, color=PALETTE["amber"], edgecolor=PALETTE["ink"], lw=0.3)
    ax.set_title(f"min-p = {pm}\nkept {k} tokens", fontsize=10)
    ax.set_xticks([])
    ax.set_xlabel("top-50 tokens")
axes[0].set_ylabel("P(token)")
plt.suptitle("Min-p: thresholded at a fraction of the leader", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Direct head-to-head: top-p vs min-p on OUR actual GPT-2 prompt.
p_top_p_real, k_top_p_real = apply_top_p(probs, 0.9)
p_min_p_real, k_min_p_real = apply_min_p(probs, 0.08)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
axes[0].bar(range(TOPN), p_top_p_real[top_idx], color=PALETTE["plum"], edgecolor=PALETTE["ink"], lw=0.3)
axes[0].set_title(f"top-p (p=0.9) on the warehouse prompt\nnucleus = {k_top_p_real} tokens")
axes[1].bar(range(TOPN), p_min_p_real[top_idx], color=PALETTE["amber"], edgecolor=PALETTE["ink"], lw=0.3)
axes[1].set_title(f"min-p (p_min=0.08) on the warehouse prompt\nkept = {k_min_p_real} tokens")
for ax in axes:
    ax.set_xticks([])
    ax.set_xlabel("top-50 tokens")
axes[0].set_ylabel("P(token)")
plt.tight_layout()
plt.show()
print(f"Both samplers picked very similar candidate sets here ({k_top_p_real} vs {k_min_p_real}).")
print("On a 'normal' mid-entropy distribution they agree. The interesting differences")
print("only show up at the extremes, which is why we need the synthetic case below.")

On well-behaved distributions, top-p and min-p produce nearly identical candidate sets — that's why people got away with top-p for so long. The arguments for switching to min-p are entirely about the *edge cases*: very sharp distributions where top-p collapses to greedy, and bimodal distributions where top-p drags in a long tail. The next cell builds the bimodal counterexample.

In [ ]:
# The pathological case top-p mishandles: bimodal distribution.
# Construct one synthetically so the failure is unambiguous.
fake = np.zeros(1000)
fake[0] = 0.50
fake[1:101] = 0.005   # 100 tokens of long flat shoulder
fake[101:] = (1.0 - 0.50 - 100*0.005) / (1000 - 101)
fake /= fake.sum()

p_top_p, k_top_p = apply_top_p(fake, 0.95)
p_min_p, k_min_p = apply_min_p(fake, 0.1)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].bar(range(150), fake[:150], color=PALETTE["indigo"])
axes[0].set_title(f"raw bimodal distribution\ntop=0.50, then a flat shoulder")
axes[1].bar(range(150), p_top_p[:150], color=PALETTE["plum"])
axes[1].set_title(f"top-p (p=0.95)\nkept {k_top_p} tokens — bad")
axes[2].bar(range(150), p_min_p[:150], color=PALETTE["amber"])
axes[2].set_title(f"min-p (p_min=0.1)\nkept {k_min_p} tokens — good")
for ax in axes:
    ax.set_xlabel("token rank")
plt.suptitle("The bimodal case where top-p degenerates", y=1.05)
plt.tight_layout()
plt.show()
print("Top-p hauls in 100 trash tokens to satisfy its mass quota.")
print("Min-p says: 'top is 10x bigger than anyone else, just take the top.'")

In [ ]:
# Composition demo: temperature + min-p together. This is the recommended
# 2026 combo. Min-p clips the trash (independent of T), then temperature
# reshapes what's left.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 1.5]):
    p_t = apply_temperature(logits, T).numpy()
    p_combo, k = apply_min_p(p_t, 0.08)
    ax.bar(range(TOPN), p_combo[top_idx], color=PALETTE["teal"], edgecolor=PALETTE["ink"], lw=0.3)
    ax.set_title(f"T={T} + min_p=0.08\nkept {k} tokens")
    ax.set_xticks([])
    ax.set_xlabel("top-50 tokens")
axes[0].set_ylabel("P(token)")
plt.suptitle("Min-p adapts to temperature: trash gets clipped at every T", y=1.05)
plt.tight_layout()
plt.show()

Notice the kept-token count *decreases* as temperature increases. That's the magic of min-p composing with temperature: when T flattens the distribution, the leader's probability drops, but every other probability drops *less proportionally*, so the relative threshold stays meaningful. Min-p is "shape-aware" in a way top-p is not.

## 6 · Generating real text

Math is fine, but you came here for outputs. Let's actually decode 200 tokens from the same prompt under each strategy and read them.

We'll measure two things on each generation:

- **Average per-step entropy** — how "uncertain" the sampler was on average. Lower = more deterministic.
- **3-gram repetition rate** — fraction of 3-grams that appear more than once. High = the model is in a loop.

In [ ]:
def sample_step(logits, strategy, **kw):
    # All strategies operate in probability space, all return the next token id.
    if strategy == "greedy":
        return int(logits.argmax()), 0.0
    if strategy == "temperature":
        p = apply_temperature(logits, kw["T"]).numpy()
    elif strategy == "top_k":
        p = apply_temperature(logits, kw.get("T", 1.0)).numpy()
        p = apply_top_k(p, kw["k"])
    elif strategy == "top_p":
        p = apply_temperature(logits, kw.get("T", 1.0)).numpy()
        p, _ = apply_top_p(p, kw["p"])
    elif strategy == "min_p":
        p = apply_temperature(logits, kw.get("T", 1.0)).numpy()
        p, _ = apply_min_p(p, kw["p_min"])
    else:
        raise ValueError(strategy)
    H = -(p * np.log2(p + 1e-12)).sum()
    return int(np.random.choice(len(p), p=p)), float(H)


def generate(prompt, strategy, n_tokens=200, seed=0, **kw):
    np.random.seed(seed)
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    entropies = []
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1].float().cpu()
        nxt, H = sample_step(logits, strategy, **kw)
        entropies.append(H)
        ids = torch.cat([ids, torch.tensor([[nxt]], device=device)], dim=1)
    text = tok.decode(ids[0])
    return text, entropies


def repetition_rate(text, n=3):
    toks = text.split()
    if len(toks) < n:
        return 0.0
    grams = [tuple(toks[i:i+n]) for i in range(len(toks) - n + 1)]
    counts = Counter(grams)
    repeats = sum(c for g, c in counts.items() if c > 1)
    return repeats / len(grams)

Two notes on the implementation above:

- We always apply temperature *first*, then truncation (top-k / top-p / min-p). This is the order vLLM, llama.cpp, and HuggingFace all use. Reversing the order is a subtle bug — truncating first and then heating up would re-spread mass into tokens you'd already decided to drop.
- We're computing the per-step entropy of the *post-sampling* distribution, not the model's raw distribution. So for greedy, entropy is exactly 0 — there's only one token with non-zero probability after the reshape.

In [ ]:
STRATEGIES = [
    ("greedy",      {}),
    ("temperature", {"T": 0.7}),
    ("temperature", {"T": 1.2}),
    ("top_k",       {"k": 40, "T": 1.0}),
    ("top_p",       {"p": 0.92, "T": 1.0}),
    ("min_p",       {"p_min": 0.08, "T": 1.0}),
]

results = []
for name, kw in STRATEGIES:
    text, ents = generate(PROMPT, name, n_tokens=200, seed=42, **kw)
    label = name + (" " + ", ".join(f"{k}={v}" for k, v in kw.items()) if kw else "")
    results.append({
        "label": label,
        "text": text,
        "avg_H": float(np.mean(ents)),
        "rep_rate": repetition_rate(text),
    })
    print(f"{label:<35}  avg H = {np.mean(ents):.2f}  3-gram rep = {repetition_rate(text):.3f}")

In [ ]:
# Print the actual generations side by side.
for r in results:
    print("=" * 80)
    print(r["label"])
    print(f"  avg per-step entropy: {r['avg_H']:.2f} bits")
    print(f"  3-gram repetition:    {r['rep_rate']:.3f}")
    print("-" * 80)
    print(r["text"][len(PROMPT):].strip())
    print()

In [ ]:
# Bar chart of the two metrics across strategies.
labels = [r["label"] for r in results]
ents = [r["avg_H"] for r in results]
reps = [r["rep_rate"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bar_colors = [PALETTE["rose"], PALETTE["amber"], PALETTE["plum"], PALETTE["teal"], PALETTE["indigo"], PALETTE["lime"]]
axes[0].barh(labels, ents, color=bar_colors, edgecolor=PALETTE["ink"])
axes[0].set_xlabel("average per-step entropy (bits)")
axes[0].set_title("How 'uncertain' was the sampler on average?")
axes[0].invert_yaxis()

axes[1].barh(labels, reps, color=bar_colors, edgecolor=PALETTE["ink"])
axes[1].set_xlabel("3-gram repetition rate")
axes[1].set_title("How loopy is the output?")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

Look at the greedy bar. Entropy is zero (it's deterministic), and the repetition rate is *much* higher than anything else. That isn't a coincidence — it's the central pathology of greedy decoding on open-ended prompts.

Top-p, top-k, and min-p with reasonable settings all sit in roughly the same low-repetition zone, with min-p typically slightly tighter than top-p at the same "feel" of diversity.

## 7 · Break it on purpose: the failure modes

Two things you should *see* fail with your own eyes so you remember them.

### 7a · T=0 (greedy) on an open-ended prompt → looping hell

GPT-2 small is a small model and falls into attractor loops easily, but every model does this — frontier models just take longer to get there. The mechanism: once the model emits a phrase, that phrase is now in the context, which makes it *more* probable to emit again, which puts it in the context twice, which... you see where this is going. Greedy has no stochastic escape valve, so once it starts looping, it never stops.

In [ ]:
LOOP_PROMPT = "Once upon a time, there was a small village by the river. Every morning"
text, _ = generate(LOOP_PROMPT, "greedy", n_tokens=200, seed=1)
print(text[len(LOOP_PROMPT):].strip())

Stare at that for a second. Find the loop. There's almost always a 6-15 token cycle that just repeats forever. The model is technically picking the most likely token at every step — it's just that the most likely continuation of "...by the river. Every morning, the villagers would go to the river" is, apparently, "and the villagers would go to the river". Greedy has no exit.

### 7b · T=2.0 → gibberish

The other end: crank temperature so high the distribution becomes basically uniform, and watch the model emit token salad.

In [ ]:
text, ents = generate(LOOP_PROMPT, "temperature", n_tokens=200, seed=1, T=2.0)
print(text[len(LOOP_PROMPT):].strip())
print(f"\navg per-step entropy: {np.mean(ents):.2f} bits  (raw model was ~{H:.2f})")

That's not "creative". That's "uniformly sampling a 50,000-element vocabulary including punctuation, byte-pair fragments, and rare unicode". Temperature inflates the long tail *equally with* the legitimate alternatives, and the long tail is much, much bigger than the legitimate alternatives. Diversity is not the same as randomness.

### 7c · The sweet spot is *narrow*

Let's sweep temperature finely and watch repetition rate transition from "loops" to "fine" to "gibberish".

In [ ]:
TEMP_SWEEP = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 1.8, 2.2]
sweep = []
for T in TEMP_SWEEP:
    if T == 0.0:
        text, ents = generate(LOOP_PROMPT, "greedy", n_tokens=150, seed=2)
    else:
        text, ents = generate(LOOP_PROMPT, "temperature", n_tokens=150, seed=2, T=T)
    sweep.append({"T": T, "rep": repetition_rate(text), "H": float(np.mean(ents))})

Ts = [s["T"] for s in sweep]
reps = [s["rep"] for s in sweep]
hs = [s["H"] for s in sweep]

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(Ts, reps, "o-", color=PALETTE["rose"], lw=2.2, label="3-gram repetition")
ax1.set_xlabel("temperature T")
ax1.set_ylabel("3-gram repetition rate", color=PALETTE["rose"])
ax1.tick_params(axis="y", labelcolor=PALETTE["rose"])
ax1.axvspan(0.7, 1.2, alpha=0.15, color=PALETTE["teal"], label="usable zone")

ax2 = ax1.twinx()
ax2.plot(Ts, hs, "s--", color=PALETTE["indigo"], lw=2, label="avg entropy (bits)")
ax2.set_ylabel("avg per-step entropy (bits)", color=PALETTE["indigo"])
ax2.tick_params(axis="y", labelcolor=PALETTE["indigo"])
ax2.spines["right"].set_visible(True)

ax1.set_title("Temperature sweep on an open-ended prompt: the U-curve of pain")
plt.tight_layout()
plt.show()

You can see the U: high repetition at T=0 (loops), low repetition in the middle, and then climbing entropy past T~1.3 as the model dissolves. The **usable zone is genuinely narrow**, maybe T ∈ [0.7, 1.2] for open-ended generation with a small model. For a 70B frontier model, the zone is shifted a bit but it's still narrow.

And critically — **for math/code/extraction tasks, T=0 is not in the bad zone**. Greedy fails on *open-ended generation* because there's no single right answer and the model gets stuck circling alternatives. On a math problem, there's one right answer, and you want it.

## 8 · A strongly-held opinion about defaults

Conventional defaults you will see in the wild:

- `temperature=0.7` — inherited from the OpenAI API ~2020. Was reasonable for InstructGPT in 2022. Has no business being your default in 2026.
- `top_p=1.0, top_k=50` — HuggingFace `generate()` defaults. Top-k=50 was a 2019 paper number. Top-p=1 means "no truncation".
- `top_p=0.9` — common community default. Fine until you hit the bimodal failure.

What you should actually use in 2026, opinionated edition:

| task type | recommendation |
|---|---|
| math / code / structured extraction / retrieval / RAG answer synthesis | `T=0` (greedy) — the right answer is the most likely one, full stop |
| chatbot, general assistant | `T=1.0` + `min_p=0.05`–`0.1`. Yes, T=1, the model's honest distribution, with min-p as a guardrail |
| creative writing, brainstorming | `T=1.0`–`1.2` + `min_p=0.05`. *Do not* use T>1.5; that's not creativity, it's noise |
| reasoning models with `<think>` traces | `T=0.6`–`0.8` + `min_p=0.05`. DeepSeek-R1 and successors recommend this. The CoT needs *some* exploration but not too much |

A few principles to internalize:

1. **Min-p > top-p > top-k**, basically always. There is no situation where top-k is the best choice.
2. **Temperature and min-p compose well**. Min-p clips the trash, then temperature reshapes what's left. Use both.
3. **Greedy is not a sin.** It's the right answer for any task with a unique correct answer. The fact that "diverse samples" sound smarter is *vibes*, not benchmarks.
4. **Defaults are sticky.** The temperature=0.7 you inherit from a tutorial will outlive three model upgrades. Pick deliberately.

Now go look at whatever sampler config is currently checked into your repo. That's the last mile, and it's usually the dirtiest mile.

## 9 · Checkpoint quiz

Five questions. Try to answer before peeking at the toggle.

**Q1.** What does temperature actually change about the next-token distribution?

<details><summary>answer</summary>

It scales the logits before softmax: $P_T = \mathrm{softmax}(z/T)$. T<1 sharpens (more mass on top tokens), T>1 flattens (more mass on the tail). It does **not** change the *ranking* of any token — only how concentrated the distribution is around the leader.
</details>

**Q2.** Why does top-k handle the "sharp distribution" case poorly?

<details><summary>answer</summary>

Because $k$ is fixed. If the model is highly confident and the top 3 tokens already cover 99% of mass, top-k=50 still keeps 47 essentially-noise tokens in the candidate set. Top-p and min-p both adapt automatically; top-k cannot.
</details>

**Q3.** Construct a distribution where top-p with p=0.95 makes a worse decision than min-p with p_min=0.1.

<details><summary>answer</summary>

The bimodal case from section 5: top token p=0.5, next 100 tokens each p≈0.005. Top-p with p=0.95 needs to accumulate 0.95 mass, so it pulls in all 100 trash tokens. Min-p threshold = 0.1 × 0.5 = 0.05, which only the top token clears. Min-p picks 1 token (correct, the model is decisive); top-p picks 101 (wrong).
</details>

**Q4.** Why does greedy decoding (T=0) loop on open-ended prompts but work fine on math problems?

<details><summary>answer</summary>

On a math problem there's a single correct continuation that dominates the distribution, so greedy picks it and moves on. On open-ended generation, several continuations are roughly equally good — once greedy commits to one, that phrase enters the context and becomes *more* likely the next time around, creating a positive-feedback loop with no stochastic escape. Greedy = no exit.
</details>

**Q5.** You're shipping a customer support agent that answers from retrieved docs. What sampler settings, and why?

<details><summary>answer</summary>

`temperature=0` (greedy), or possibly `T=0.3` + `min_p=0.1` if you want a sliver of diversity. The task has *correct answers* — they're in the retrieved context. You don't want creativity; you want the model to surface the right answer. The "interesting" continuations a higher temperature would produce are exactly the wrong thing for RAG: they're the model exploring the tail of "things that sound plausible but aren't in the docs". Aka hallucinations.
</details>

## 10 · Bridge to Module 20

Every strategy in this notebook reshapes the distribution **right before sampling**. They are last-mile interventions: by the time you reach the sampler, the model has already decided what it thinks. You're just choosing which of its opinions to honor.

Modules 20 and 21 are about a much earlier intervention: changing the model so the distribution it produces is *better* in the first place.

- **Module 20 — RLHF & DPO**: take a pretrained model and *teach it* what a good answer looks like, by training against human preferences (RLHF) or directly against pairs of preferred/rejected outputs (DPO). The sampler doesn't change; the distribution coming out of the model does.
- **Module 21 — GRPO & RLVR**: the 2025 wave. Use *verifiable* rewards (math answers, code tests) instead of human preferences, and group-relative advantages instead of a value network. This is what trained DeepSeek-R1.

Sampling is the last mile. Alignment is the first mile. You need both — and you cannot fix bad alignment with clever sampling, or vice versa.

See you in Module 20.